<a href="https://colab.research.google.com/github/AndresMontesDeOca/NLP_1/blob/main/Desafios/Desafio__1_AndresMontesDeOca.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Procesamiento de Lenguaje Natural I
# **Desafío 1**



In [13]:
# %pip install numpy scikit-learn

### Vectorización de texto y modelo de clasificación Naïve Bayes con el dataset 20 newsgroups

In [14]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.naive_bayes import MultinomialNB, ComplementNB
from sklearn.metrics import f1_score

Utilizamos **20newsgroups** por ser un dataset clásico de NLP ya viene incluido y formateado en sklearn

In [15]:
from sklearn.datasets import fetch_20newsgroups
import numpy as np

## Carga de datos

Cargamos los datos (ya separados de forma predeterminada en train y test)

El dataset 20 Newsgroups contiene aproximadamente 18 000 publicaciones de grupos de noticias distribuidas en 20 temas. Está dividido en dos subconjuntos: uno para entrenamiento (train set) y otro para pruebas (test set).

In [16]:
newsgroups_train = fetch_20newsgroups(subset='train', remove=('headers', 'footers', 'quotes'))
newsgroups_test = fetch_20newsgroups(subset='test', remove=('headers', 'footers', 'quotes'))
print(f"Cantidad real de documentos en train: {len(newsgroups_train.data)}")
print(f"Cantidad real de documentos en test: {len(newsgroups_test.data)}")
print("Atributos del objeto:", newsgroups_train.keys())

Cantidad real de documentos en train: 11314
Cantidad real de documentos en test: 7532
Atributos del objeto: dict_keys(['data', 'filenames', 'target_names', 'target', 'DESCR'])


## Vectorización

Instanciamos un vectorizador.

Podemos ver diferentes parámetros de instanciación en la documentación de sklearn https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.TfidfVectorizer.html

In [17]:
tfidfvect = TfidfVectorizer()

En el atributo `data` accedemos al texto

In [18]:
print(newsgroups_train.data[0])

I was wondering if anyone out there could enlighten me on this car I saw
the other day. It was a 2-door sports car, looked to be from the late 60s/
early 70s. It was called a Bricklin. The doors were really small. In addition,
the front bumper was separate from the rest of the body. This is 
all I know. If anyone can tellme a model name, engine specs, years
of production, where this car is made, history, or whatever info you
have on this funky looking car, please e-mail.


Con la interfaz habitual de sklearn podemos ajustar el vectorizador (obtener el vocabulario y calcular el vector IDF) y transformar directamente los datos.

Podemos denominar `X_train` como la matriz documento-término.

In [19]:
X_train = tfidfvect.fit_transform(newsgroups_train.data)

Recordemos que las vectorizaciones por conteos son de tipo sparse, por ello sklearn convenientemente devuelve los vectores de documentos como matrices de tipo sparse.

In [20]:

print(type(X_train))
print(f'shape: {X_train.shape}')
print(f'Cantidad de documentos: {X_train.shape[0]}')
print(f'Tamaño del vocabulario (dimensionalidad de los vectores): {X_train.shape[1]}')

<class 'scipy.sparse._csr.csr_matrix'>
shape: (11314, 101631)
Cantidad de documentos: 11314
Tamaño del vocabulario (dimensionalidad de los vectores): 101631


Una vez ajustado el vectorizador, podemos acceder a atributos como el vocabulario aprendido. Es un diccionario que va de términos a índices.

El índice es la posición en el vector de documento.

In [21]:
tfidfvect.vocabulary_['car']

25775

Probamos con una palbra que no está en el documento.

In [22]:
tfidfvect.vocabulary_['cocoliso']

KeyError: 'cocoliso'

Es muy útil tener el diccionario opuesto que va de índices a términos

In [23]:
idx2word = {v: k for k,v in tfidfvect.vocabulary_.items()}

En `y_train` guardamos los targets que son enteros

In [24]:
y_train = newsgroups_train.target
y_train[:10]

array([ 7,  4,  4,  1, 14, 16, 13,  3,  2,  4])

Hay 20 clases correspondientes a los 20 grupos de noticias

In [25]:
print(f'clases {np.unique(newsgroups_test.target)}')
newsgroups_test.target_names

clases [ 0  1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19]


['alt.atheism',
 'comp.graphics',
 'comp.os.ms-windows.misc',
 'comp.sys.ibm.pc.hardware',
 'comp.sys.mac.hardware',
 'comp.windows.x',
 'misc.forsale',
 'rec.autos',
 'rec.motorcycles',
 'rec.sport.baseball',
 'rec.sport.hockey',
 'sci.crypt',
 'sci.electronics',
 'sci.med',
 'sci.space',
 'soc.religion.christian',
 'talk.politics.guns',
 'talk.politics.mideast',
 'talk.politics.misc',
 'talk.religion.misc']

## Similaridad de documentos

Veamos similaridad de documentos. Tomemos algún documento

In [26]:
idx = 4811
print(newsgroups_train.data[idx])

THE WHITE HOUSE

                  Office of the Press Secretary
                   (Pittsburgh, Pennslyvania)
______________________________________________________________
For Immediate Release                         April 17, 1993     

             
                  RADIO ADDRESS TO THE NATION 
                        BY THE PRESIDENT
             
                Pittsburgh International Airport
                    Pittsburgh, Pennsylvania
             
             
10:06 A.M. EDT
             
             
             THE PRESIDENT:  Good morning.  My voice is coming to
you this morning through the facilities of the oldest radio
station in America, KDKA in Pittsburgh.  I'm visiting the city to
meet personally with citizens here to discuss my plans for jobs,
health care and the economy.  But I wanted first to do my weekly
broadcast with the American people. 
             
             I'm told this station first broadcast in 1920 when
it reported that year's presidential elec

Medimos la similaridad coseno con todos los documentos de train

In [27]:
cossim = cosine_similarity(X_train[idx], X_train)[0]

Podemos ver los valores de similaridad ordenados de mayor a menor

In [28]:
np.sort(cossim)[::-1]

array([1.        , 0.70930477, 0.67474953, ..., 0.        , 0.        ,
       0.        ])

Después vemos a qué documentos corresponden

In [29]:
np.argsort(cossim)[::-1]

array([4811, 6635, 4253, ..., 9019, 9016, 8748])

Obtenemos los 5 documentos más similares:

In [30]:
mostsim = np.argsort(cossim)[::-1][1:6]
print(mostsim)

[6635 4253 3596 4271 3746]


El documento original pertenece a la clase:

In [31]:
newsgroups_train.target_names[y_train[idx]]

'talk.politics.misc'

Revisamos las clases de los 5 más similares:

In [32]:
for i in mostsim:
  print(newsgroups_train.target_names[y_train[i]])

talk.politics.misc
talk.politics.misc
talk.politics.misc
talk.politics.misc
talk.politics.misc


### Modelo de clasificación Naïve Bayes

Instanciamos el modelo de clasificación Naive Bayes y lo entrenamos con sklearn

In [33]:
clf = MultinomialNB()
clf.fit(X_train, y_train)

MultinomialNB()

Ya tenemos nuestro vectorizador ya ajustado en train, vectorizamos los textos
del conjunto de test.

In [34]:
X_test = tfidfvect.transform(newsgroups_test.data)
y_test = newsgroups_test.target
y_pred =  clf.predict(X_test)

El F1-score es una métrica adecuada para evaluar el desempeño de modelos de clasificación, especialmente cuando existe desbalance entre clases.

* El promediado macro calcula el promedio del F1-score de cada clase, otorgando el mismo peso a todas las clases.
* El promediado micro calcula las métricas de forma global considerando todas las predicciones; en problemas de clasificación multiclase suele ser equivalente a la accuracy, por lo que no es la mejor métrica cuando el dataset está desbalanceado.

In [35]:
f1_score(y_test, y_pred, average='macro')

0.5854345727938506

---

## **Consigna del Desafío 1**
**Cada experimento realizado debe estar acompañado de una explicación o interpretación de lo observado.**



**1. Vectorizar documentos**
* Tomar 5 documentos al azar y medir similaridad con el resto de los documentos.
Estudiar los 5 documentos más similares de cada uno analizar si tiene sentido
la similaridad según el contenido del texto y la etiqueta de clasificación.





In [47]:
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

def analizar_similitud(idx):
    print("-" * 80)
    print(f"Documento original (Índice {idx}) - Clase: {newsgroups_train.target_names[y_train[idx]]}")
    print(f"Texto original (primeros 200 caracteres):\n{newsgroups_train.data[idx][:200]}...")
    print("\nTop 5 documentos más similares:")

    cossim = cosine_similarity(X_train[idx], X_train)[0]
    mostsim = np.argsort(cossim)[::-1][1:6]

    for i, sim_idx in enumerate(mostsim):
        sim_score = cossim[sim_idx]
        sim_class = newsgroups_train.target_names[y_train[sim_idx]]
        print(f"  {i+1}. Índice: {sim_idx} | Clase: {sim_class} | Similitud: {sim_score:.4f}")
        # Mostrar los primeros 150 caracteres del documento similar, limpiando saltos de línea
        snippet = newsgroups_train.data[sim_idx][:150].replace('\n', ' ')
        print(f"     Texto: {snippet}...")
    print("-" * 80)


# Seleccionar 5 índices al azar
np.random.seed(1515) # Fijamos semilla para reproducibilidad
random_indices = np.random.choice(X_train.shape[0], 5, replace=False)
print(f"Índices seleccionados: {random_indices}\n")


Índices seleccionados: [5841 3940 9846 7384 3046]



- Creo una funcion para hacer las comparaciones, para que quede mas prolijo
- Imporimo los indices de las 5 documentos elegidos al azar

In [49]:
# Caso 1
analizar_similitud(random_indices[0])

--------------------------------------------------------------------------------
Documento original (Índice 5841) - Clase: sci.electronics
Texto original (primeros 200 caracteres):

We'll let you live, but just this once....

There's more to a real "storage" scope than just a long-persistence
phosphor.  Actually, the phosphor ISN'T usually anything special at all;
what makes a s...

Top 5 documentos más similares:
  1. Índice: 6938 | Clase: sci.electronics | Similitud: 0.2936
     Texto: Greetings. I've been seeing the word "storage" mentioned 	around oscilliscopes but I'm curious, what does it mean?  	If my life depended on it, I'd sa...
  2. Índice: 10226 | Clase: comp.graphics | Similitud: 0.2521
     Texto: Hi to all out there.  We have this problem, and I'm not certain I'm solving it in the correct way.  I was wondering if anyone can shed light on this, ...
  3. Índice: 10697 | Clase: sci.electronics | Similitud: 0.2270
     Texto:  You are correct; the fastest "complete" image th

- Documento 5841: Electronica
- Los documentos mas similares hablan tambien sobre electronica, pero talvez de topicos mas puntuales (hardware, cryptografia)
- Por eso los valores de similitud no son tan altos como vimos en clase (0.7), en este casi varian entre 0.2 y 0.3

In [50]:
# Caso 2
analizar_similitud(random_indices[1])

--------------------------------------------------------------------------------
Documento original (Índice 3940) - Clase: rec.motorcycles
Texto original (primeros 200 caracteres):

1 hr/drink for the first 4 drinks.
1.5 hours/drink for the next 6 drinks.
2 hours/drink for the rest....

Top 5 documentos más similares:
  1. Índice: 4398 | Clase: rec.motorcycles | Similitud: 0.2719
     Texto:  Go bikeless.  You drink and drive, you pay. No smiley. ...
  2. Índice: 1778 | Clase: rec.motorcycles | Similitud: 0.1802
     Texto: :  What is a general rule of thumb for sobriety and cycling? Couple hours after : you "feel" sober? What? Or should I just work with "If I drink tonig...
  3. Índice: 2520 | Clase: rec.motorcycles | Similitud: 0.1756
     Texto:  In my case it goes down after the first four, because the fifth one usually makes me throw up the last two.  Needless to say, I don't drink very much...
  4. Índice: 8908 | Clase: rec.motorcycles | Similitud: 0.1753
     Texto:   I took an 

- Documento 3940: Motocicletas
- Los cuatro documentos mas similares tambien estan relacionados con el motociclismo
- El quinto documento es un documento medico, no relacionado con motocicletas. Supongo que el valor de la similitud es alto porque en el texto habla sobre valores los valores de altotid en ciertas ciudades, y esto se podria estar confundiendo con las cilindradas de las motos

In [51]:
# Caso 3
analizar_similitud(random_indices[2])

--------------------------------------------------------------------------------
Documento original (Índice 9846) - Clase: comp.graphics
Texto original (primeros 200 caracteres):
I have a small program to extract a 640x480 image from a vga 16 color screen,
and store that image in a TIFF file.  I need to insert the image into a
sales brochure, which I then need printed in 4 col...

Top 5 documentos más similares:
  1. Índice: 4166 | Clase: comp.graphics | Similitud: 0.3327
     Texto: Archive-name: graphics/resources-list/part2 Last-modified: 1993/04/17   Computer Graphics Resource Listing : WEEKLY POSTING [ PART 2/3 ] =============...
  2. Índice: 336 | Clase: comp.os.ms-windows.misc | Similitud: 0.2741
     Texto: Does anybody have any idea where I could find a program that can convert a .GIF image into a .BMP image suitable for a Windows  wallpaper (i.e. 256 co...
  3. Índice: 3451 | Clase: comp.os.ms-windows.misc | Similitud: 0.2572
     Texto: Hi there, I'm having a bizarre video p

- Documento 9846: Graficos
- Tanto el documento original como los comparativos son muy tecnicos, hablan sobre resoluciones y formatos de archivos de imagenes
- Algunos estan claisificados como de graficos (al igual que el documento original) y otros a temas relacionados con el Sistema Operativo Windows, aunque hablan sobre temas muy puntuales de como Widndows maneja los archivos de imagenes o las placas de video

In [52]:
# Caso 4
analizar_similitud(random_indices[3])

--------------------------------------------------------------------------------
Documento original (Índice 7384) - Clase: sci.med
Texto original (primeros 200 caracteres):
SP> From: paulson@tab00.larc.nasa.gov (Sharon Paulson)
SP> to describe here.  I have a fourteen year old daugter who experienced
SP> a seizure on November 3, 1992 at 6:45AM after eating Kellog's Frost...

Top 5 documentos más similares:
  1. Índice: 6315 | Clase: sci.med | Similitud: 0.4772
     Texto: I am posting to this group in hopes of finding someone out there in network newsland who has heard of something similar to what I am going to describe...
  2. Índice: 7002 | Clase: sci.med | Similitud: 0.2352
     Texto: :  {much deleted] :  :  : The fact that this happened while eating two sugar coated cereals made : by Kellog's makes me think she might be having an a...
  3. Índice: 5204 | Clase: sci.med | Similitud: 0.2314
     Texto:     Path: news.larc.nasa.gov!darwin.sura.net!zaphod.mps.ohio-state.edu!uwm.edu!li

- Indice 7384: Ciencias Medicas
- Aca los documentos similares tambien hablan sobre medicina, mas que nada sobre algun problema relacionado cuando si ingieren copos de maiz azucarados.
- 4/5 casos son escritos como en un foro, consultando a la comunidad. El documento original era un email, al parecer no detecta los textos medicos como similares

In [54]:
# Caso 5
analizar_similitud(random_indices[4])

--------------------------------------------------------------------------------
Documento original (Índice 3046) - Clase: comp.graphics
Texto original (primeros 200 caracteres):

I've got the 6.0 spec (obviously since I quoted it in my last posting). 
My gripe about TIFF is that it's far too complicated and nearly
infinitely easier to write than to read, which I think hurts y...

Top 5 documentos más similares:
  1. Índice: 3158 | Clase: comp.graphics | Similitud: 0.5237
     Texto: :  : >I've got the 6.0 spec (obviously since I quoted it in my last posting).  : >My gripe about TIFF is that it's far too complicated and nearly : >i...
  2. Índice: 16 | Clase: comp.graphics | Similitud: 0.4288
     Texto:  I certainly do use it whenever I have to do TIFF, and it usually works very well.  That's not my point.  I'm >philosophically< opposed to it because ...
  3. Índice: 2810 | Clase: comp.graphics | Similitud: 0.3717
     Texto:  I'm sure it is, and I am not amused.  Every time I read th

- 3046: Graficos
- En este caso los cinco documentos mas similares, tambien son de la misma clase Graficos
- Lo interesante aqui es que el documento mas parecido, pareciera ser un Reply del documento original, que es un post. Y esto tiene todo el sentido, ya que parte del texto es exactamente igual


**2. Construir un modelo de clasificación por prototipos (tipo zero-shot).**
* Clasificar los documentos de un conjunto de test comparando cada uno con todos los de entrenamiento y asignar la clase al label del documento del conjunto de entrenamiento con mayor similaridad.

In [55]:
from sklearn.metrics import f1_score
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

# Calculamos la matriz de similitud coseno entre todo test y todo train
# El resultado es una matriz de dimensiones (n_test, n_train)
print("Calculando similitud coseno entre test y train...")
similarity_matrix = cosine_similarity(X_test, X_train)

# Para cada documento en test, encontramos el índice del documento de train con mayor similitud
# axis=1 indica que buscamos el máximo a lo largo de las columnas (los documentos de train)
best_match_indices = np.argmax(similarity_matrix, axis=1)

# Asignamos la clase del documento de train más similar a las predicciones de test
y_pred_zero_shot = y_train[best_match_indices]

# Evaluamos el desempeño del modelo zero-shot
f1_macro_zs = f1_score(y_test, y_pred_zero_shot, average='macro')
print(f"F1-Score Macro (Clasificación por prototipos): {f1_macro_zs:.4f}")


Calculando similitud coseno entre test y train...
F1-Score Macro (Clasificación por prototipos): 0.5050


- Buscamos para cada documento de Test, cual es el mas parecido (usando la similitud coseno) en train, y le asignamos esa clase.
- Utilizando un modelo por prototipos (sin entrenamiento) obtuvimos un F1 de 0.5, lo cual es aceptable

**3. Entrenar modelos de clasificación Naïve Bayes para maximizar el desempeño de clasificación**

* F1-Score Macro en el conjunto de datos de test. Considerar cambiar parámetros
de instanciación del vectorizador y los modelos y probar modelos de Naïve Bayes Multinomial y ComplementNB.

**NO cambiar el hiperparámetro ngram_range de los vectorizadores**.

In [58]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB, ComplementNB
from sklearn.metrics import f1_score

# 1. Ajustar parámetros del vectorizador (sin modificar ngram_range)
# Usamos stop_words para eliminar palabras muy comunes en inglés.
# max_df=0.5: ignorar términos que aparecen en más del 50% de los documentos.
# min_df=5: ignorar términos que aparecen en menos de 5 documentos.
tfidf_opt = TfidfVectorizer(stop_words='english', max_df=0.5, min_df=5)

X_train_opt = tfidf_opt.fit_transform(newsgroups_train.data)
X_test_opt = tfidf_opt.transform(newsgroups_test.data)

print(f"Tamaño del nuevo vocabulario: {X_train_opt.shape[1]}")
print("-" * 50)

# 2. Entrenar y evaluar MultinomialNB
# Modificamos el parámetro alpha (suavizado de Laplace)
mnb = MultinomialNB(alpha=0.1)
mnb.fit(X_train_opt, y_train)
y_pred_mnb = mnb.predict(X_test_opt)
f1_mnb = f1_score(y_test, y_pred_mnb, average='macro')
print(f"F1-Score Macro (MultinomialNB optimizado): {f1_mnb:.4f}")

# 3. Entrenar y evaluar ComplementNB
# ComplementNB suele funcionar mejor que MultinomialNB en datasets desbalanceados
cnb = ComplementNB(alpha=0.1)
cnb.fit(X_train_opt, y_train)
y_pred_cnb = cnb.predict(X_test_opt)
f1_cnb = f1_score(y_test, y_pred_cnb, average='macro')
print(f"F1-Score Macro (ComplementNB optimizado): {f1_cnb:.4f}")


Tamaño del nuevo vocabulario: 17797
--------------------------------------------------
F1-Score Macro (MultinomialNB optimizado): 0.6772
F1-Score Macro (ComplementNB optimizado): 0.6745


- Agregaremos parametros como stop_words='english' (para quitar palabras muy comunes) y definiremos valores para max_df y min_df (para ignorar palabras que aparecen en casi todos los documentos o en muy pocos)
- Se pasaron de mas de 100k palabras en el vocabulario, a menos de 18k, ayudando a remover el ruido de palabras que no aportan ni ayudan a la clasificacion
- Agregamos el parametro alpha en los modelos de NB, ya que como vimos en clase podriamos tener problemas si en el conjunto de test el modelo encuentra una palabra que nuca vio en train (Fix de Laplace)
- Con todas estas transformaciones logramos subir la performance del F1 de 0.5 a 0.67

**4. Transponer la matriz documento-término.**
* De esa manera se obtiene una matriz término-documento que puede ser interpretada como una colección de vectorización de palabras.
* Estudiar ahora similaridad entre palabras tomando 5 palabras y estudiando sus 5 más similares.

**Elegir las palabras MANUALMENTE para evitar la aparición de términos poco interpretables**.

In [68]:
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

# Definimos una función para buscar las palabras más similares y no repetir código
def buscar_similares(palabra, n=5):
    if palabra not in word2idx:
        print(f"La palabra '{palabra}' no se encuentra en el vocabulario.")
        return

    print("-" * 50)
    print(f"Palabra objetivo: '{palabra.upper()}'")
    word_idx = word2idx[palabra]
    similitudes = cosine_similarity(X_term_doc[word_idx], X_term_doc)[0]
    top_indices = np.argsort(similitudes)[::-1][1:n+1]

    for i, idx in enumerate(top_indices):
        print(f"  {i+1}. {idx2word_opt[idx]} (Similitud: {similitudes[idx]:.4f})")


# 1. Transponer la matriz documento-término para obtener término-documento
X_term_doc = X_train_opt.T
print(f"Dimensiones de la matriz término-documento: {X_term_doc.shape}\n")

# Obtener el vocabulario (término -> índice) y el inverso (índice -> término)
word2idx = tfidf_opt.vocabulary_
idx2word_opt = {idx: word for word, idx in word2idx.items()}

Dimensiones de la matriz término-documento: (17797, 11314)



In [69]:
buscar_similares('car')

--------------------------------------------------
Palabra objetivo: 'CAR'
  1. cars (Similitud: 0.1923)
  2. dealer (Similitud: 0.1773)
  3. civic (Similitud: 0.1634)
  4. loan (Similitud: 0.1560)
  5. owner (Similitud: 0.1484)


- Car: Relacion semantica bien clara, nos muestra palabras relacionada con autos o marcas

In [70]:
buscar_similares('computer')

--------------------------------------------------
Palabra objetivo: 'COMPUTER'
  1. shopper (Similitud: 0.1349)
  2. verlag (Similitud: 0.1248)
  3. delicate (Similitud: 0.1196)
  4. drive (Similitud: 0.1105)
  5. hackers (Similitud: 0.1082)


- Computer: Aca la relacion semantica no parece muy clara, y se ve como el valor de al similitud coseno es considerablmente mas baja en el punto anterior (cars)

In [63]:
buscar_similares('disease')

--------------------------------------------------
Palabra objetivo: 'DISEASE'
  1. diagnosed (Similitud: 0.3139)
  2. herman (Similitud: 0.2820)
  3. patients (Similitud: 0.2571)
  4. diseases (Similitud: 0.2433)
  5. genetic (Similitud: 0.1980)


- Disease: Aca la relacion semantica con las palabras mas parecidas es clara, todas hablan sobre enfermedades o salud en general. Los valores de la similitud coseno son muy altos

In [74]:
buscar_similares('soccer')

--------------------------------------------------
Palabra objetivo: 'SOCCER'
  1. mia (Similitud: 0.2261)
  2. symposium (Similitud: 0.2167)
  3. entertain (Similitud: 0.2077)
  4. lion (Similitud: 0.2008)
  5. rabbis (Similitud: 0.1981)


- Soccer: Para el football no se ven relaciones semanticas claras con las top 5 que encontro. Seguramente las ultimas dos son equipos locales de USA, ademas la relaciono con entretain, y no con otra palabra relacionada con el deporte. Talvez eso indique como se vive el football en USA, solo como un entretenimiento

In [75]:
buscar_similares('space')

--------------------------------------------------
Palabra objetivo: 'SPACE'
  1. nasa (Similitud: 0.3178)
  2. shuttle (Similitud: 0.2784)
  3. exploration (Similitud: 0.2328)
  4. aeronautics (Similitud: 0.2219)
  5. cfa (Similitud: 0.2164)


- Space: Tambien al ser muy tecnica, encuentra palabras similares y con valores de similitud coseno altos.

**Conclusión Final:**
Transponer la matriz TF-IDF funciona de manera excelente para capturar el **significado semántico** subyacente de las palabras, basándose pura y exclusivamente en sus estadísticas de co-ocurrencia a lo largo de los documentos de los 20 grupos de noticias.